# 03 — Evaluation playground

The research workflow, hands-on. Everything here runs the **exact live code** over
recorded data (same `SymbolPipeline`, same features, same label queue), so results
transfer to live paper trading by construction.

## The workflow

```
scripts/record.py      -> capture live quotes into DuckDB   (or reuse a soak DB)
scripts/replay.py      -> detailed single-config evaluation
scripts/experiment.py  -> grid: models x horizons x sessions (+ trade sim)
scripts/report.py      -> report + PNG from any live/paper session DB
```

CLI equivalents of what we do below:
```bash
.venv/bin/python scripts/experiment.py --dbs data/session.duckdb \
    --symbols BTC/USD --horizons 10 30 --non-overlapping --fee-bps 0.2
```

In [1]:
from pathlib import Path
import sys
import numpy as np, pandas as pd
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
from signals.evaluation import evaluate

DB = str(ROOT / "data" / "session.duckdb")

## 1. Single detailed run

`evaluate()` returns per-symbol scores; quartiles show whether quality holds up
over the session (walk-forward: the model at Q4 has learned from Q1–Q3).

In [2]:
r = await evaluate(DB, ["BTC/USD"], model_kind="classifier", horizon_s=10.0)
score = r.symbols["BTC/USD"]
pd.DataFrame([{"Q": i+1, "n": s.n, "dir": round(s.dir_acc, 3),
               "persistence": round(s.dir_persistence, 3),
               "edge%": round(s.edge_pct, 1)}
              for i, s in enumerate(score.quartiles())])

,Q,n,dir,persistence,edge%
0,1,990,0.846,0.833,20.8
1,2,991,0.601,0.779,2.4
2,3,991,0.709,0.845,8.9
3,4,990,0.680,0.780,7.7


## 2. The number that matters: simulated net PnL after costs

`simulate_trading` applies the live threshold rule (enter only when |prediction| >
fee + half-spread + dead-zone), holds one position at a time for the horizon, and
charges the full spread + two fees per round trip. Compare venue cost scenarios —
this is the quantitative case for the equities pivot:

In [3]:
rows = []
for label, fee in [("crypto-ish (5bps/side)", 5.0), ("equity-ish (0.2bps/side)", 0.2)]:
    for kind in ("hoeffding", "classifier"):
        r = await evaluate(DB, ["BTC/USD"], kind, 10.0)
        sim = r.symbols["BTC/USD"].simulate_trading(r.horizon_ns, fee_bps=fee)
        rows.append({"costs": label, "model": kind, "trades": sim.trades,
                     "hit": round(sim.hit_rate, 2) if sim.trades else None,
                     "net_bps": round(sim.net_bps_sum, 1)})
pd.DataFrame(rows)

,costs,model,trades,hit,net_bps
0,crypto-ish (5bps/side),hoeffding,2,0.00,-33.4
1,crypto-ish (5bps/side),classifier,0,NaN,0.0
2,equity-ish (0.2bps/side),hoeffding,22,0.09,-125.1
3,equity-ish (0.2bps/side),classifier,0,NaN,0.0


A model with no real edge should ideally trade **zero** times — refusing to play
a losing game is a feature. The regressor trades and loses; the classifier mostly
abstains. Neither has demonstrated positive expectancy yet (tiny data!) — the point
is that the harness now *measures* expectancy honestly.

## 3. Feature ablation

Which features carry the information? Knock groups out via `FeatureConfig` and
re-run. (With ~30min of data this is illustrative — rerun on the 48h soak DB for
real answers.)

In [4]:
from signals.features.engine import FeatureConfig

configs = {
    "default (lags 1,2,4,8)": None,
    "short memory (lag 1 only)": FeatureConfig(lag_returns=(1,)),
    "long memory (lags 4,8,16,32)": FeatureConfig(lag_returns=(4, 8, 16, 32)),
}
rows = []
for name, cfg in configs.items():
    r = await evaluate(DB, ["BTC/USD"], "classifier", 10.0,
                       feature_config=cfg, non_overlapping=True)
    seg = r.symbols["BTC/USD"].overall()
    rows.append({"features": name, "n": seg.n, "dir": round(seg.dir_acc, 3),
                 "persistence": round(seg.dir_persistence, 3),
                 "edge%": round(seg.edge_pct, 1)})
pd.DataFrame(rows)

,features,n,dir,persistence,edge%
0,"default (lags 1,2,4,8)",138,0.496,0.504,-4.4
1,short memory (lag 1 only),138,0.496,0.504,-4.8
2,"long memory (lags 4,8,16,32)",138,0.496,0.504,-4.4


## 4. Extending the system

- **Add a feature**: one O(1) accumulator + one `raw[...] =` line in
  `src/signals/features/engine.py` (z-normalization is automatic). Update the
  key-set test in `tests/test_engine.py`.
- **Add a model**: new `kind` branch in `src/signals/model/online.py` — anything
  exposing `predict_one`/`learn_one` (all of River) drops in.
- **Change the trade rule**: `src/signals/signal/policy.py` (live) and
  `simulate_trading` in `src/signals/evaluation.py` (research) — keep them in sync.

## 5. Ground rules (from docs/RESEARCH.md)

1. Independent (non-overlapping) windows for any claim.
2. Beat **both** baselines (zero-MAE and sign-persistence) or it's noise.
3. The final arbiter is net paper PnL over weeks, not any backtest statistic.

Research roadmap and current findings: [docs/RESEARCH.md](../docs/RESEARCH.md).